# Create a FullPrompt for a new game

This tutorial defines a hypothetical private-signal choice prompt without using an existing game helper. Architecture: concrete `PromptBlock` values → immutable `PrivateSignalChoiceFullPrompt` → `CompiledPrompt` → normalized `CompletionRequest` → provider. Production classes used below live in `mas_cc.prompts`, `mas_cc.core`, `mas_cc.config`, `mas_cc.llm_providers`, and `mas_cc.planning`.

In [10]:
from __future__ import annotations
import json, os
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field
from typing import Any
from mas_cc.config import LLMProviderConfig
from mas_cc.core import MessageRole, ValidationIssue
from mas_cc.llm_providers import (CompletionRequest, OfflinePricingSource, UniversityPricingSource, create_llm_provider)
from mas_cc.planning import LogicalCallSpec, static_preflight
from mas_cc.prompts import UNBOUND, FullPrompt, PromptBlock, RegexTokenCounter, ResponseContract, Unbound


## 1. Define all six concrete blocks
Each block owns its value validation and deterministic rendering. Fixed description/rules are bound at construction; required dynamic values start as `UNBOUND`; the optional hint may remain `UNBOUND`.

In [11]:
def issue(name: str, message: str, value: Any) -> tuple[ValidationIssue, ...]:
    return (ValidationIssue(f'prompt.blocks.{name}.value', message, value),)

@dataclass(frozen=True, slots=True)
class DescriptionBlock(PromptBlock[str]):
    name: str = field(init=False, default='description')
    title: str = field(init=False, default='Description')
    role: MessageRole = field(init=False, default=MessageRole.SYSTEM)
    value: str | Unbound = 'Use your private signal to choose one public action.'
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default='fixed')
    def value_issues(self, value): return () if isinstance(value, str) and value.strip() else issue(self.name, 'must be non-empty text', value)
    def render(self): return str(self.value)

@dataclass(frozen=True, slots=True)
class RulesBlock(PromptBlock[tuple[str, ...]]):
    name: str = field(init=False, default='rules')
    title: str = field(init=False, default='Rules')
    role: MessageRole = field(init=False, default=MessageRole.SYSTEM)
    value: tuple[str, ...] | Unbound = ('Choose exactly one available action.', 'Do not invent observations.')
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default='fixed')
    def value_issues(self, value): return () if isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and all(isinstance(x, str) and x for x in value) else issue(self.name, 'must contain non-empty strings', value)
    def render(self): return '\n'.join(f'{i}. {x}' for i, x in enumerate(self.value, 1))

@dataclass(frozen=True, slots=True)
class AvailableActionsBlock(PromptBlock[tuple[str, ...]]):
    name: str = field(init=False, default='available_actions')
    title: str = field(init=False, default='Available actions')
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: tuple[str, ...] | Unbound = UNBOUND
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default='dynamic')
    def value_issues(self, value): return () if isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and len(value) >= 2 and len(set(value)) == len(value) else issue(self.name, 'must contain at least two unique actions', value)
    def render(self): return 'Available actions: ' + ', '.join(self.value)

@dataclass(frozen=True, slots=True)
class PrivateSignalBlock(PromptBlock[str]):
    name: str = field(init=False, default='private_signal')
    title: str = field(init=False, default='Private signal')
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: str | Unbound = UNBOUND
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default='dynamic')
    sensitive: bool = field(init=False, default=True)
    def value_issues(self, value): return () if isinstance(value, str) and value.strip() else issue(self.name, 'must be non-empty text', value)
    def render(self): return f'Your private signal is: {self.value}'

@dataclass(frozen=True, slots=True)
class VisibleMemoryBlock(PromptBlock[tuple[Mapping[str, Any], ...]]):
    name: str = field(init=False, default='visible_memory')
    title: str = field(init=False, default='Visible memory')
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: tuple[Mapping[str, Any], ...] | Unbound = UNBOUND
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default='dynamic')
    sensitive: bool = field(init=False, default=True)
    def value_issues(self, value): return () if isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and all(isinstance(x, Mapping) for x in value) else issue(self.name, 'must be a sequence of mappings', value)
    def render(self): return 'Visible memory is empty.' if not self.value else 'Visible memory: ' + json.dumps([dict(x) for x in self.value], sort_keys=True)

@dataclass(frozen=True, slots=True)
class OptionalHintBlock(PromptBlock[str]):
    name: str = field(init=False, default='optional_hint')
    title: str = field(init=False, default='Optional hint')
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: str | Unbound = UNBOUND
    required: bool = field(init=False, default=False)
    binding: str = field(init=False, default='dynamic')
    def value_issues(self, value): return () if isinstance(value, str) else issue(self.name, 'must be text', value)
    def render(self): return f'Optional hint: {self.value}'


## 2. Define the authoritative FullPrompt order and response contract

In [12]:
class PrivateSignalChoiceFullPrompt(FullPrompt):
    def concrete_prompt_type(self) -> str:
        return 'private_signal_choice'

prompt_definition = PrivateSignalChoiceFullPrompt(
    family='private_signal_choice', version=1,
    blocks=(DescriptionBlock(), RulesBlock(), AvailableActionsBlock(), PrivateSignalBlock(), VisibleMemoryBlock(), OptionalHintBlock()),
    response_contract=ResponseContract('choice_only', ('red', 'blue')),
    message_mode='merge_consecutive_roles', block_separator='\n\n',
)
assert [b.name for b in prompt_definition.blocks] == ['description', 'rules', 'available_actions', 'private_signal', 'visible_memory', 'optional_hint']
assert not prompt_definition.validate().is_valid


## 3. Empty versus unbound, immutable two-agent binding, and non-leakage

In [13]:
agent_one = prompt_definition.bind(available_actions=('red', 'blue'), private_signal='red-biased', visible_memory=())
agent_two = prompt_definition.bind(available_actions=('red', 'blue'), private_signal='blue-biased', visible_memory=({'choice': 'blue'},), optional_hint='Recent success can matter.')
assert agent_one is not agent_two
assert agent_one.block('private_signal').value == 'red-biased'
assert agent_two.block('private_signal').value == 'blue-biased'
assert agent_one.block('visible_memory').render() == 'Visible memory is empty.'
assert 'optional_hint' in agent_one.compile().omitted_blocks
assert not prompt_definition.validate().is_valid  # unbound memory is a failure, empty bound memory is not


## 4. Inspect blocks, normalized messages, token estimates, and fingerprints

In [14]:
counter = RegexTokenCounter()
compiled_one = agent_one.compile(counter)
compiled_two = agent_two.compile(counter)
block_table = compiled_one.blocks_as_dicts()
message_table = compiled_one.messages_as_dicts()
assert compiled_one.total_tokens == sum(row['token_count'] for row in block_table)
assert compiled_one.definition_hash == compiled_two.definition_hash
assert compiled_one.instance_hash != compiled_two.instance_hash
assert agent_one.compile(counter).instance_hash == compiled_one.instance_hash
print(json.dumps({'blocks': block_table, 'messages': message_table, 'block_tokens': compiled_one.total_tokens, 'message_tokens': compiled_one.message_token_total, 'definition_hash': compiled_one.definition_hash, 'instance_hash': compiled_one.instance_hash}, indent=2))


{
  "blocks": [
    {
      "name": "description",
      "title": "Description",
      "role": "system",
      "version": 1,
      "order": 1,
      "content": "Use your private signal to choose one public action.",
      "token_count": 10
    },
    {
      "name": "rules",
      "title": "Rules",
      "role": "system",
      "version": 1,
      "order": 2,
      "content": "1. Choose exactly one available action.\n2. Do not invent observations.",
      "token_count": 15
    },
    {
      "name": "available_actions",
      "title": "Available actions",
      "role": "user",
      "version": 1,
      "order": 3,
      "content": "Available actions: red, blue",
      "token_count": 6
    },
    {
      "name": "private_signal",
      "title": "Private signal",
      "role": "user",
      "version": 1,
      "order": 4,
      "content": "Your private signal is: red-biased",
      "token_count": 8
    },
    {
      "name": "visible_memory",
      "title": "Visible memory",
      "role"

## 5. Validate local responses and construct one normalized request

In [15]:
assert compiled_one.response_contract.validate('red').is_valid
assert not compiled_one.response_contract.validate('I choose red').is_valid
request = CompletionRequest(messages=compiled_one.messages, temperature=0.0, max_output_tokens=16, seed=1026, metadata={'prompt_family': compiled_one.family, 'prompt_version': compiled_one.version, 'definition_hash': compiled_one.definition_hash, 'instance_hash': compiled_one.instance_hash})
university_request = request
openai_request = CompletionRequest(messages=request.messages, temperature=request.temperature, max_output_tokens=request.max_output_tokens, seed=request.seed, metadata=request.metadata)
assert university_request.wire_messages() == openai_request.wire_messages()


## 6. Live controls and secret-safe provider configuration
The committed controls visibly default to true. Automated validation replaces them in memory with false; it does not edit this notebook.

In [25]:
USE_LIVE_UNIVERSITY_PRICING = True
CALL_UNIVERSITY = True
CALL_OPENAI = True
university_config = LLMProviderConfig(type='university', 
                                      model=os.getenv('POTSDAM_MODEL', 'microsoft/gpt-5.4-nano'),
                                      credentials_env='POTSDAM_API_KEY', 
                                      base_url_env='BASE_POTSDAM_LLM_URL', 
                                      max_output_tokens=1000)
openai_config = LLMProviderConfig(type='openai', model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'), credentials_env='OPENAI_API_KEY', max_output_tokens=16)
credential_status = {name: bool(os.getenv(name)) for name in ('POTSDAM_API_KEY', 'BASE_POTSDAM_LLM_URL', 'OPENAI_API_KEY', 'POTSDAM_MODEL', 'OPENAI_MODEL')}
print({'credential_variables_configured': credential_status})


{'credential_variables_configured': {'POTSDAM_API_KEY': True, 'BASE_POTSDAM_LLM_URL': True, 'OPENAI_API_KEY': False, 'POTSDAM_MODEL': False, 'OPENAI_MODEL': False}}


## 7. University live availability/pricing and OpenAI auditable preflight

In [26]:
university_quote = None
try:
    if USE_LIVE_UNIVERSITY_PRICING:
        university_quote = UniversityPricingSource(university_config).fetch('university', university_config.model)
        university_preflight = static_preflight(university_request, university_config, LogicalCallSpec(1), pricing_quote=university_quote, assumed_output_tokens=8, explicit_override=False)
        print({'university_pricing_status': university_quote.status, 'launch_status': university_preflight.launch_status})
    else:
        print({'university_live_preflight': 'disabled for non-network validation'})
except Exception as exc:
    print({'university_preflight_error_type': type(exc).__name__})
#openai_quote = OfflinePricingSource().fetch('openai', openai_config.model)
#openai_preflight = static_preflight(openai_request, openai_config, LogicalCallSpec(1), pricing_quote=openai_quote, assumed_output_tokens=8, explicit_override=True)
#print({'openai_pricing_status': openai_quote.status, 'launch_status': openai_preflight.launch_status})


{'university_pricing_status': 'known', 'launch_status': 'permitted'}


## 8. University completion (independent cell)

In [28]:
provider = create_llm_provider(university_config)
await provider._ensure_endpoint()
import requests
resp = requests.post(
    provider._chat_url,
    headers={"Authorization": f"Bearer {provider._key}", "Content-Type": "application/json"},
    json={
        "model": provider.model,
        "messages": university_request.wire_messages(),
        "temperature": university_request.temperature,
        "max_tokens": university_request.max_output_tokens,
    },
    timeout=30,
)
print(resp.status_code, resp.text)
provider.close()


400 {"error":{"message":"litellm.UnsupportedParamsError: gpt-5 models (including gpt-5-codex) don't support temperature=0.0. Only temperature=1 is supported. For gpt-5.1, temperature is supported when reasoning_effort='none' (or not specified, as it defaults to 'none'). To drop unsupported params set `litellm.drop_params = True`. Received Model Group=microsoft/gpt-5.4-nano\nAvailable Model Group Fallbacks=None","type":"None","param":null,"code":"400"}}


In [23]:
university_summary = {'status': 'disabled'}
if CALL_UNIVERSITY:
    provider = None
    try:
        provider = create_llm_provider(university_config)
        response = await provider.complete(university_request)
        actual = None if university_quote is None or university_quote.pricing is None else university_quote.pricing.cost(input_tokens=response.usage.input_tokens, output_tokens=response.usage.output_tokens, cached_input_tokens=response.usage.cached_input_tokens or 0)
        university_summary = {'status': 'completed', 'provider': response.provider, 'model': response.model, 'usage': response.usage.to_dict(), 'actual_cost': None if actual is None else actual.to_dict()}
    except Exception as exc:
        university_summary = {'status': 'failed-safely', 'error_type': type(exc).__name__}
    finally:
        if provider is not None: provider.close()
print(university_summary)


{'status': 'failed-safely', 'error_type': 'ProviderError'}


## 9. OpenAI completion (independent cell)

In [ ]:
openai_summary = {'status': 'disabled'}
if CALL_OPENAI:
    provider = None
    try:
        provider = create_llm_provider(openai_config)
        response = await provider.complete(openai_request)
        actual = None if openai_quote.pricing is None else openai_quote.pricing.cost(input_tokens=response.usage.input_tokens, output_tokens=response.usage.output_tokens, cached_input_tokens=response.usage.cached_input_tokens or 0)
        openai_summary = {'status': 'completed', 'provider': response.provider, 'model': response.model, 'usage': response.usage.to_dict(), 'actual_cost': None if actual is None else actual.to_dict()}
    except Exception as exc:
        openai_summary = {'status': 'failed-safely', 'error_type': type(exc).__name__}
    finally:
        if provider is not None: provider.close()
print(openai_summary)


## 10. Promotion-to-production checklist

1. Move the six blocks and `PrivateSignalChoiceFullPrompt` to `games/private_signal_choice/prompts.py`.
2. Bind a new prompt from each immutable private observation; never share a bound prompt across agents.
3. Register the family/version factory and a schema-v2 prompt component.
4. Add empty/representative/maximum bound planning scenarios.
5. Add golden message, validation, fingerprint, privacy, retry, and provider-boundary tests.
6. Store only local detailed traces; redact sensitive values from public manifests.